In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by checking whether the model returned any `function_call` items.

- It sends the current `messages` history to the model.
- If the response includes a function call, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- At the end of the iteration, if `has_function_calls == False`, it breaks out of the `while True` loop.

So the stop condition is: **no function calls in the model’s response**.


In [2]:
%pip install opentelemetry-api opentelemetry-sdk


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter(out=sys.stdout))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [12]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        # 1. Call the parent class constructor
        super().__init__(*args, **kwargs) 
        
    def rag_traced(self, query):

        
        with tracer.start_as_current_span("rag") as span:
            search_results = self.search_traced(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm_traced(prompt)
            span.set_attribute("rag", "rag")
            return response.output_text
            

    def search_traced(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            self.search(query, num_results)
            span.set_attribute("search", "search")

    def llm_traced(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            self.llm(prompt)
            span.set_attribute("llm", "llm")


In [13]:
from starter import index, client
rag_traced = RAGTraced(index=index, llm_client=client)


In [14]:
#Q1
q1_query = "How does the agentic loop keep calling the model until it stops?"
q1_answer = rag_traced.rag_traced(q1_query)
print(q1_answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xeacacfaa7a6a27495aa0be09a559f2e8",
        "span_id": "0x97da9c67e249546b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xcaed15126319a4c8",
    "start_time": "2026-07-24T22:00:03.132586Z",
    "end_time": "2026-07-24T22:00:03.137176Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search": "search"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2dbabbdd-5c40-424c-bed5-8ccc735c42a5",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0xeacacfaa7a6a27495aa0be09a559f2e8",
        "span_id": "0xcaed15126319a4c8",
        "trace_state": "

TypeError: 'NoneType' object is not iterable

In [7]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0x9c8f4481bc9aa8540b440cd88c830894",
        "span_id": "0xc74d92e47bfedabb",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-24T21:55:50.386789Z",
    "end_time": "2026-07-24T21:55:50.386816Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2dbabbdd-5c40-424c-bed5-8ccc735c42a5",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
